# Домашнее задание №4

Кто проходил курс GPU прекрасно знают датасет. Данные находятся в файле Train_Set_90621.csv Amount Defaulted - эту переменную нужно удалить=)

Что ожидается? - творчество)

1) Начните с анализа баланса классов, пропусков, статистик при помощи DataFrame API
2) Посомтрите статистики, заполните пропуски при помощи уже MLlib
3) Соберите пайплайн, похожий на наш, где будет обработка данных, обучение моделей и все при помощи Spark
4) Разбейте данные на train/test + реализуйте подбор параметров одним из способов спарка
5) Cохраниет пайплайн на диск
6) Проверьте качество модели на отложенной test выборке

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler, Imputer
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("homework")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
sc = spark.sparkContext

## Анализ данных

In [3]:
df = spark.read.csv("Train_Set_90621.csv", header=True, inferSchema=True)

In [4]:
df.printSchema()

root
 |-- Application ID: integer (nullable = true)
 |-- Bank Masked: string (nullable = true)
 |-- Bank Type: string (nullable = true)
 |-- Approved_Timestamp: string (nullable = true)
 |-- Name Masked: string (nullable = true)
 |-- Business Owner State: string (nullable = true)
 |-- Business_Industry_Type_Code: integer (nullable = true)
 |-- Approved_Year: integer (nullable = true)
 |-- New Business: integer (nullable = true)
 |-- Term: integer (nullable = true)
 |-- BankState: string (nullable = true)
 |-- Interest Rate: integer (nullable = true)
 |-- Employees: integer (nullable = true)
 |-- Gross Disbursed Amount: integer (nullable = true)
 |-- Term_years: integer (nullable = true)
 |-- Jobs Retained: integer (nullable = true)
 |-- Male to Female Employees Ratio: integer (nullable = true)
 |-- Expected Company Income: long (nullable = true)
 |-- Funds available with company: integer (nullable = true)
 |-- Gross_Apprv_Amount: integer (nullable = true)
 |-- Company Branch Code: inte

In [5]:
df.show(5, truncate=False)

+--------------+--------------+---------+------------------+--------------------+--------------------+---------------------------+-------------+------------+----+-------------+-------------+---------+----------------------+----------+-------------+------------------------------+-----------------------+----------------------------+------------------+-------------------+-------------+--------------+--------------------+------------------+------------+----------------+-------------------+--------------+
|Application ID|Bank Masked   |Bank Type|Approved_Timestamp|Name Masked         |Business Owner State|Business_Industry_Type_Code|Approved_Year|New Business|Term|BankState    |Interest Rate|Employees|Gross Disbursed Amount|Term_years|Jobs Retained|Male to Female Employees Ratio|Expected Company Income|Funds available with company|Gross_Apprv_Amount|Company Branch Code|City or Rural|Jobs Generated|Carry-forward Credit|Documents Provided|Balance Left|Amount Defaulted|Final_Appved_Amount|Defa

In [6]:
df.count()

195118

In [7]:
df = df.drop("Application ID", "Approved_Timestamp", "Approved_Year")

---

In [8]:
df.groupby("Bank Masked").count().orderBy(F.col("count").desc()).show(10)

+--------------+-----+
|   Bank Masked|count|
+--------------+-----+
|  Bank_Name_42| 9133|
|   Bank_Name_3| 6688|
|Bank_Number_13| 6021|
|  Bank_Name_11| 4903|
|Bank_Number_37| 4436|
|  Bank_Name_17| 3756|
|  Bank_Name_49| 3687|
|Bank_Number_49| 3398|
|  Bank_Name_63| 2841|
| Bank_Number_9| 2525|
+--------------+-----+
only showing top 10 rows


In [9]:
df = df.drop("Bank Masked")  # высококардинальный признак

---

In [10]:
df.groupby("Bank Type").count().orderBy(F.col("count").desc()).show()

+---------+------+
|Bank Type| count|
+---------+------+
|  Private|130150|
|     Govt| 64888|
|     NULL|    80|
+---------+------+



---

In [11]:
df.groupby("Name Masked").count().orderBy(F.col("count").desc()).show(10)

+------------------+-----+
|       Name Masked|count|
+------------------+-----+
|  Business_Name_39|   91|
| Business_Name_355|   30|
|Business_Name_1807|   28|
|Business_Name_2404|   28|
|Business_Name_2602|   27|
|Business_Name_3812|   26|
|Business_Name_2487|   25|
|Business_Name_1296|   22|
|Business_Name_7395|   22|
|Business_Name_4874|   21|
+------------------+-----+
only showing top 10 rows


In [12]:
df = df.drop("Name Masked")  # высококардинальный признак

---

In [13]:
df.groupby("Business Owner State").count().orderBy(F.col("count").desc()).show(10)

+--------------------+-----+
|Business Owner State|count|
+--------------------+-----+
|               Delhi|58546|
|             Haryana|56421|
|         Maharashtra|48088|
|       Uttar Pradesh|24167|
|              Orissa| 2603|
|          Tamil Nadu| 2040|
|         Uttarakhand| 1293|
|           Karnataka|  798|
|              Punjab|  616|
|              Kerala|  543|
+--------------------+-----+
only showing top 10 rows


In [14]:
top_states = ["Delhi", "Haryana", "Maharashtra", "Uttar Pradesh"]

df = df.withColumn(
    "Business Owner State",
    F.when(F.col("Business Owner State").isin(top_states), F.col("Business Owner State"))
    .otherwise("Other")
)

In [15]:
df.groupby("Business Owner State").count().orderBy(F.col("count").desc()).show()

+--------------------+-----+
|Business Owner State|count|
+--------------------+-----+
|               Delhi|58546|
|             Haryana|56421|
|         Maharashtra|48088|
|       Uttar Pradesh|24167|
|               Other| 7896|
+--------------------+-----+



---

In [16]:
df.groupby("Business_Industry_Type_Code").count().orderBy(F.col("count").desc()).show(10)

+---------------------------+-----+
|Business_Industry_Type_Code|count|
+---------------------------+-----+
|                       1231|29967|
|                       1230|14194|
|                     723341| 4170|
|                     723442| 2880|
|                     812342| 2152|
|                     622441| 2021|
|                     723340| 1909|
|                     813343| 1511|
|                     625641| 1465|
|                     723441| 1402|
+---------------------------+-----+
only showing top 10 rows


In [17]:
df = df.drop("Business_Industry_Type_Code")  # высококардинальный признак

---

In [18]:
df.groupby("New Business").count().orderBy(F.col("count").desc()).show(10)

+------------+------+
|New Business| count|
+------------+------+
|           1|140163|
|           2| 54703|
|           0|   216|
|        NULL|    36|
+------------+------+



---

In [19]:
df.groupby("BankState").count().orderBy(F.col("count").desc()).show(10)

+-------------+-----+
|    BankState|count|
+-------------+-----+
|        Delhi|74016|
|  Maharashtra|60322|
|      Haryana|39331|
|Uttar Pradesh|16083|
|       Orissa| 1996|
|   Tamil Nadu|  949|
|  Uttarakhand|  798|
|    Karnataka|  671|
|       Punjab|  468|
|       Kerala|  319|
+-------------+-----+
only showing top 10 rows


In [20]:
top_states = ["Delhi", "Haryana", "Maharashtra", "Uttar Pradesh"]

df = df.withColumn(
    "BankState",
    F.when(F.col("BankState").isin(top_states), F.col("BankState"))
    .otherwise("Other")
)

## Статистики и заполнение пропусков

In [21]:
df = df.fillna({"Bank Type": "Unknown", "New Business": 0})

In [22]:
df = df.withColumn("New Business", F.col("New Business").cast("string"))

## Pipeline

In [23]:
num_features = ["Term"]

for feature in num_features:
    df = df.withColumn(feature, F.col(feature).astype("double"))

cat_features = [
    "Bank Type",
    "Business Owner State",
    "New Business",
    "BankState",
]
indexes = [f"{feature}_idx" for feature in cat_features]
ohed_features = [f"{feature}_OHEd" for feature in cat_features]


indexers = [StringIndexer(inputCol=feature, outputCol=index) for feature, index in zip(cat_features, indexes)]

ohe_encoder = OneHotEncoder(inputCols=indexes, outputCols=ohed_features)

vector_assembler = VectorAssembler(inputCols=ohed_features+num_features, outputCol="features")

gbt = GBTClassifier(featuresCol="features", labelCol="Default_Status", seed=42)


pipeline = Pipeline(stages=[
    *indexers,
    ohe_encoder,
    vector_assembler,
    gbt,
])

## HyperParameter Tunning

In [24]:
df_train, df_test = df.randomSplit([0.8, 0.2], seed=42)

In [25]:
param_grid = ParamGridBuilder().addGrid(gbt.maxDepth, [3, 4, 5]).build()

evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="Default_Status", metricName="areaUnderROC")

crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=2,
    parallelism=2,
)

In [26]:
crossval_model = crossval.fit(df_train)

## Save

In [27]:
crossval_model.write().overwrite().save("model")

## Evaluation

In [28]:
y_pred_train = crossval_model.transform(df_train)
y_pred_test = crossval_model.transform(df_test)

In [29]:
evaluator.evaluate(y_pred_train)

0.7936547306223635

In [30]:
evaluator.evaluate(y_pred_test)

0.7903526878486212